### PhayaThaiBERT(เพิ่ม) + Cross Validation(ต่อ)

In [32]:
!pip install --upgrade "accelerate>=1.1.0" transformers torch

Defaulting to user installation because normal site-packages is not writeable
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached torch-2.12.0-cp313-cp313-win_amd64.whl.metadata (31 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
Using cached torch-2.12.0-cp313-cp313-win_amd64.whl (123.0 MB)
Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)

  Attempting uninstall: setuptools

    Found existing installation: setuptools 82.0.0

    Uninstalling setuptools-82.0.0:

   ---------------------------------------- 0/3 [setuptools]
   ---------------------------------------- 0/3 [setuptools]
   ---------------------------------------- 0/3 [setuptools]
   ---------------------------------------- 0/3 [setuptools]
   ---------------------------------------- 0/3 [setuptools]
   ---------------------------------------- 0/3 [setuptools]
   ---------------------------------------- 0/3

ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import classification_report,accuracy_score, f1_score
import gc
import numpy as np


from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

from datasets import Dataset, DatasetDict, load_from_disk

In [39]:
model_name = "clicknext/phayathaibert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [40]:
full_ds = load_from_disk("../model")
print(full_ds)

DatasetDict({
    train: Dataset({
        features: ['Label', 'input_ids', 'attention_mask'],
        num_rows: 433
    })
    validation: Dataset({
        features: ['Label', 'input_ids', 'attention_mask'],
        num_rows: 108
    })
})


In [42]:
labels = np.array(full_ds["train"]["Label"])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X=np.zeros(len(labels)), y=labels)):

    print(f"Fold {fold+1} / 5")

    train_fold = full_ds["train"].select(train_idx)
    val_fold   = full_ds["train"].select(val_idx)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,                   
        num_labels=2,                 
        ignore_mismatched_sizes=True  
    )

    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        return_tensors="pt"
    )

    training_args = TrainingArguments(
        output_dir=f"./phayathaibert_fold_{fold+1}",
        num_train_epochs=3,            
        learning_rate=2e-5,         
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",         
        save_strategy="epoch",         
        load_best_model_at_end=True,    
        metric_for_best_model="eval_f1",  # เลือกตาม F1
        greater_is_better=True,
        logging_steps=10,
        seed=42,
        report_to="none"
        
    )

    
    def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
            "f1": f1_score(labels, preds, average="binary")
    }

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_fold,
        eval_dataset=val_fold,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()
# Evaluate
    predictions = trainer.predict(val_fold)
    preds = np.argmax(predictions.predictions, axis=-1)
    true_labels = predictions.label_ids

    acc = accuracy_score(true_labels, preds)
    print(f"\n{'='*40}")
    print(f"Fold {fold+1} - Accuracy: {acc:.4f}")

    print("\nClassification Report:")
    print(classification_report(
            true_labels, preds,
            target_names=["Normal (0)", "Scam (1)"],
            zero_division=0))
                                
    fold_results.append({
    "fold": fold + 1,
    "accuracy": acc
    })         

# Save
    trainer.save_model()
    print(f"Model saved to ./phayathaibert_fold_{fold+1}")
                     
# ล้าง memory
    del model, predictions, preds, true_labels
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(fold_results)
print(results_df)
print(f"\nMean Accuracy: {results_df['accuracy'].mean():.2f} +/- {results_df['accuracy'].std():.2f}")

IndentationError: expected an indented block after function definition on line 41 (2937318605.py, line 42)